Environment Setup

In [1]:
import sys
import os

if 'google.colab' in sys.modules:
    print("Detected Google Colab. Bypassing dependency resolver...")

    !pip install timm
    !pip install git+https://github.com/IBM/terratorch.git
else:
    print("Running locally. Assuming repository is already downloaded.")

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(os.path.abspath(os.getcwd()))

Detected Google Colab. Bypassing dependency resolver...
  Cloning https://github.com/IBM/terratorch.git to /tmp/pip-req-build-_cbavv2p
  Running command git clone --filter=blob:none --quiet https://github.com/IBM/terratorch.git /tmp/pip-req-build-_cbavv2p
  Resolved https://github.com/IBM/terratorch.git to commit d1267fbd278cd23bc05a1c22cc077e18481dd7c0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

Note: Please restart the Colab session before proceeding to avoid dependency conflicts.

In [1]:
print("\nCloning the official repository...")
!git clone https://github.com/karol140903/terramind-adversarial-robustness.git

%cd terramind-adversarial-robustness


Cloning the official repository...
Cloning into 'terramind-adversarial-robustness'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 163 (delta 58), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 30.23 MiB | 11.07 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/terramind-adversarial-robustness


Global Setup & Imports

In [2]:
import torch
import pandas as pd
from tqdm.notebook import tqdm
from terratorch import BACKBONE_REGISTRY

# Import core modules
from src.attacks import pgd_cos
from src.metrics import compute_metrics
from src.utils import set_seed

# Lock the global environment for reproducibility
set_seed(42)
print("Global environment locked with fixed seed (42).")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing computations on: {device}")

Global environment locked with fixed seed (42).
Executing computations on: cuda


Transferability Experiment (Source to Targets)

In [3]:
# Experiment Configuration
SOURCE_MODEL = "terramind_v1_base"
TARGET_MODELS = ["terramind_v1_tiny", "terramind_v1_small", "terramind_v1_large"]

PATCH_PATHS = [
    {"id": "04_city", "path": "data/patches/patch_04_city.pt"},
    {"id": "09_water", "path": "data/patches/patch_09_water.pt"},
    {"id": "18_forest", "path": "data/patches/patch_18_forest.pt"}
]

EPSILON = 0.01
PGD_ALPHA = 0.002
PGD_STEPS = 10

# Dictionary to hold the adversarial examples generated by the source model
adversarial_payloads = {}
transfer_results = []

# ==========================================
# PHASE 1: Generate Noise on Source Model
# ==========================================
print(f"\n--- PHASE 1: Crafting Adversarial Examples on {SOURCE_MODEL.upper()} ---")
set_seed(42)

source_model = BACKBONE_REGISTRY.build(SOURCE_MODEL, pretrained=True, modalities=["S2L2A"])
source_model.eval()
source_model = source_model.to(device)

for patch_info in tqdm(PATCH_PATHS, desc="Generating Source Noise"):
    patch_id = patch_info["id"]
    tensor_path = patch_info["path"]

    # Load original tensor
    x_orig = torch.load(tensor_path, weights_only=True).to(device)

    # Execute PGD Attack
    set_seed(42)
    x_adv = pgd_cos(source_model, x_orig, epsilon=EPSILON, alpha=PGD_ALPHA, steps=PGD_STEPS)

    # Evaluate white-box impact on source model
    cos_sim, l2_dist = compute_metrics(source_model, x_orig, x_adv)

    # Save the crafted payload to memory for black-box testing later
    adversarial_payloads[patch_id] = {
        "orig": x_orig.cpu(),
        "adv": x_adv.cpu()
    }

    transfer_results.append({
        "Source_Model": SOURCE_MODEL,
        "Target_Model": SOURCE_MODEL,
        "Patch_ID": patch_id,
        "Transfer_Type": "White-Box (Baseline)",
        "Transferred_Cosine": cos_sim,
        "Transferred_L2": l2_dist
    })

# Clean up VRAM
del source_model
torch.cuda.empty_cache()

# ==========================================
# PHASE 2: Test Transferability on Targets
# ==========================================
print("\n--- PHASE 2: Testing Black-Box Transferability ---")

for target_name in TARGET_MODELS:
    set_seed(42)

    target_model = BACKBONE_REGISTRY.build(target_name, pretrained=True, modalities=["S2L2A"])
    target_model.eval()
    target_model = target_model.to(device)

    for patch_id, payload in tqdm(adversarial_payloads.items(), desc=f"Testing on {target_name}"):

        # Load the pre-crafted tensors to the device
        x_orig = payload["orig"].to(device)
        x_adv_transferred = payload["adv"].to(device)

        # Evaluate how the target model reacts to the source's noise
        set_seed(42)
        cos_sim, l2_dist = compute_metrics(target_model, x_orig, x_adv_transferred)

        transfer_results.append({
            "Source_Model": SOURCE_MODEL,
            "Target_Model": target_name,
            "Patch_ID": patch_id,
            "Transfer_Type": "Black-Box",
            "Transferred_Cosine": cos_sim,
            "Transferred_L2": l2_dist
        })

    del target_model
    torch.cuda.empty_cache()

# ==========================================
# PHASE 3: Export Results
# ==========================================
import os
os.makedirs("results", exist_ok=True)
csv_path = "results/03_Transferability_Results.csv"

df_transfer = pd.DataFrame(transfer_results)
df_transfer.to_csv(csv_path, index=False)
print(f"\nTransferability experiment completed. Results saved to {csv_path}")

display(df_transfer)


--- PHASE 1: Crafting Adversarial Examples on TERRAMIND_V1_BASE ---


TerraMind_v1_base.pt: reconstructing file:   0%|          |  0.00B / 1.52GB            

TerraMind_v1_base.pt: downloading bytes:           |  0.00B            

Generating Source Noise:   0%|          | 0/3 [00:00<?, ?it/s]


--- PHASE 2: Testing Black-Box Transferability ---


...

TerraMind_v1_large.pt: reconstructing file:   0%|          |  0.00B / 3.79GB            

TerraMind_v1_large.pt: downloading bytes:           |  0.00B            

Testing on terramind_v1_large:   0%|          | 0/3 [00:00<?, ?it/s]


Transferability experiment completed. Results saved to results/03_Transferability_Results.csv


,Source_Model,Target_Model,Patch_ID,Transfer_Type,Transferred_Cosine,Transferred_L2
0,terramind_v1_base,terramind_v1_base,04_city,White-Box (Baseline),0.235284,583.816650
1,terramind_v1_base,terramind_v1_base,09_water,White-Box (Baseline),0.113857,667.530701
2,terramind_v1_base,terramind_v1_base,18_forest,White-Box (Baseline),0.251409,570.529358
3,terramind_v1_base,terramind_v1_tiny,04_city,Black-Box,0.953132,36.760826
4,terramind_v1_base,terramind_v1_tiny,09_water,Black-Box,0.929050,47.764118
5,terramind_v1_base,terramind_v1_tiny,18_forest,Black-Box,0.955760,35.963036
6,terramind_v1_base,terramind_v1_small,04_city,Black-Box,0.880163,127.099266
7,terramind_v1_base,terramind_v1_small,09_water,Black-Box,0.759789,181.348740
8,terramind_v1_base,terramind_v1_small,18_forest,Black-Box,0.905557,112.731979
9,terramind_v1_base,terramind_v1_large,04_city,Black-Box,0.857044,199.366913
